<a href="https://colab.research.google.com/github/alfnihilus/alfnihilus/blob/main/pre_simulated_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install biopython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 24.6 MB/s eta 0:00:00


In [4]:
# ==========================================
# 1. IMPORT & KONFIGURASI
# ==========================================
from Bio import SeqIO, Entrez
from Bio.Seq import Seq
from Bio.SeqUtils import gc_fraction, molecular_weight
from Bio.SeqUtils.ProtParam import ProteinAnalysis
from Bio.SeqRecord import SeqRecord
import pandas as pd
import re

Entrez.email = "email_kamu@example.com"

# ==========================================
# DATABASE ASAM AMINO
# ==========================================
# ubah formatnya ke Dictionary {Simbol: {Detail}} agar pencarian instan
AA_DATABASE = {
    "A": {"nama": "Alanine", "sifat": "Non-polar"},
    "R": {"nama": "Arginine", "sifat": "Basa (+)"},
    "N": {"nama": "Asparagine", "sifat": "Polar"},
    "D": {"nama": "Aspartic Acid", "sifat": "Asam (-)"},
    "C": {"nama": "Cysteine", "sifat": "Sulfur"},
    "Q": {"nama": "Glutamine", "sifat": "Polar"},
    "E": {"nama": "Glutamic Acid", "sifat": "Asam (-)"},
    "G": {"nama": "Glycine", "sifat": "Kecil"},
    "H": {"nama": "Histidine", "sifat": "Basa (+)"},
    "I": {"nama": "Isoleucine", "sifat": "Non-polar"},
    "L": {"nama": "Leucine", "sifat": "Non-polar"},
    "K": {"nama": "Lysine", "sifat": "Basa (+)"},
    "M": {"nama": "Methionine", "sifat": "Start/Sulfur"},
    "F": {"nama": "Phenylalanine", "sifat": "Aromatik"},
    "P": {"nama": "Proline", "sifat": "Kaku"},
    "S": {"nama": "Serine", "sifat": "Polar"},
    "T": {"nama": "Threonine", "sifat": "Polar"},
    "W": {"nama": "Tryptophan", "sifat": "Aromatik"},
    "Y": {"nama": "Tyrosine", "sifat": "Aromatik"},
    "V": {"nama": "Valine", "sifat": "Non-polar"},
}

# ==========================================
# 2. FUNGSI-FUNGSI ANALISIS
# ==========================================

def cari_semua_kodon(dna_string):
    starts = [m.start() for m in re.finditer('ATG', str(dna_string))]
    stops = [m.start() for m in re.finditer('TAA|TAG|TGA', str(dna_string))]
    return starts, stops

def deteksi_mutasi(ref, sampel):
    ref_str, sam_str = str(ref).upper(), str(sampel).upper()
    mutasi = []
    # Bandingkan karakter per karakter
    for i, (b1, b2) in enumerate(zip(ref_str, sam_str)):
        if b1 != b2:
            mutasi.append(f"Pos {i+1}: {b1}->{b2}")
    return mutasi if mutasi else ["Tidak ditemukan mutasi"]

def temukan_orf_terpanjang(dna_obj, table_id=1):
    sekuens_target = [dna_obj, dna_obj.reverse_complement()]
    orf_ditemukan = []
    for strand in sekuens_target:
        for frame in range(3):
            n = len(strand)
            dna_frame = strand[frame : n - (n - frame) % 3]
            protein_seq = dna_frame.translate(table=table_id)

            for fragmen in protein_seq.split('*'):
                if 'M' in fragmen:
                    orf_ditemukan.append(fragmen[fragmen.find('M'):])

    return max(orf_ditemukan, key=len) if orf_ditemukan else Seq("")

# ==========================================
# 3. PIPELINE UTAMA
# ==========================================

def jalankan_pipeline_lengkap():
    print("=== AUTOMATED PIPELINE ===")

    # --- PILIH TABEL TRANSLASI ---
    print("\n[1] TIPE ORGANISME")
    print("1. Standard (Eukariota/Umum)")
    print("11. Bakteri, Archaea, & Plastida")
    try:
        table_id = int(input("Pilih nomor tabel (1/11): "))
    except:
        table_id = 1 # Default ke Standar jika salah ketik

    # --- METODE INPUT ---
    print("\n[2] METODE INPUT")
    print("1. Manual | 2. File FASTA | 3. NCBI")
    pil = input("Pilih metode: ")

    if pil == '1':
        id_s = "Manual_Input"
        dna_raw = input("Masukkan sekuens DNA: ")
    elif pil == '2':
        fn = input("Nama file .fasta: ")
        rec = SeqIO.read(fn, "fasta"); id_s = rec.id; dna_raw = str(rec.seq)
    elif pil == '3':
        acc = input("Masukkan Accession ID: ")
        h = Entrez.efetch(db="nucleotide", id=acc, rettype="fasta", retmode="text")
        rec = SeqIO.read(h, "fasta"); id_s = rec.id; dna_raw = str(rec.seq)
    else: return

    # --- VALIDASI & REVERSE COMPLEMENT ---
    dna_bersih = dna_raw.upper().strip().replace(" ", "")
    if not set(dna_bersih).issubset(set("ATCG")):
        print("❌ Error: Karakter ilegal ditemukan!"); return

    my_dna = Seq(dna_bersih)
    rev_comp = my_dna.reverse_complement()

    # --- TRANSKRIPSI  ---
    my_rna = my_dna.transcribe()

    # --- TRANSLASI ---
    dna_bersih = dna_raw.upper().strip().replace(" ", "")
    my_dna = Seq(dna_bersih)
    my_rna = my_dna.transcribe()

    # Translasi Lengkap
    protein_lengkap = my_dna.translate(table=table_id)

    # Protein Terpanjang
    protein_orf = temukan_orf_terpanjang(my_dna, table_id=table_id)

    # --- ANALISIS KIMIA & KODON ---
    gc = gc_fraction(my_dna) * 100
    starts, stops = cari_semua_kodon(my_dna)

    # Analisis Protein (pI & Komposisi)
    if len(protein_orf) > 0:
        pa = ProteinAnalysis(str(protein_orf))
        pi = pa.isoelectric_point(); berat = molecular_weight(protein_orf, "protein")
        berat = molecular_weight(protein_orf, "protein")
        komposisi = pa.count_amino_acids()
    else:
        print("❌ Error: Tidak ditemukan ORF!"); return

    # --- MUTASI (Opsional) ---
    cek_m = input("\nIngin cek mutasi terhadap referensi? (y/n): ").lower()
    laporan_mutasi = "Tidak dicek"
    if cek_m == 'y':
        ref_input = input("Masukkan DNA Referensi: ")
        laporan_mutasi = deteksi_mutasi(ref_input, dna_bersih)

    # ==========================================
    # 4. TAMPILAN HASIL
    # ==========================================
    print("\n" + "="*50)
    print(f"LAPORAN ANALISIS SEKUENS: {id_s}")
    print("="*50)

    print(f"DNA Asli          : {str(dna_bersih)[:60]}...")
    print(f"Reverse Complement: {str(rev_comp)[:60]}...")
    print(f"RNA (Transkripsi) : {str(my_rna)[:60]}...")
    print(f"Protein Lengkap   : {str(protein_lengkap)[:60]}...")
    print(f"Protein Terpanjang (ORF): {str(protein_orf)[:60]}...")

    print(f"\nPersentase GC     : {gc*100:.2f}%")
    print(f"Berat Protein     : {berat:.2f} Da")
    print(f"Titik Isolistrik  : {pi:.2f} (pH)")

    print(f"\nLokasi Start (ATG): {(starts)}")
    print(f"Lokasi Stop       : {(stops)}")

    print("\nKomposisi Asam Amino (Jumlah):")
    # Hanya menampilkan asam amino yang ada saja agar tidak penuh
    for simbol, jumlah in komposisi.items():
        if jumlah > 0:
            # Mengambil detail dari AA_DATABASE, jika tidak ada beri nilai default
            detail = AA_DATABASE.get(simbol, {"nama": "Unknown", "sifat": "Unknown"})
            nama_aa = detail["nama"]
            sifat_aa = detail["sifat"]
            print(f"    - {nama_aa} ({simbol}) [{sifat_aa}]: {jumlah}")

    print(f"\nDaftar Mutasi     : {laporan_mutasi}")
    print("="*50)

    # --- OPSI SIMPAN ---
    simpan = input("\nSimpan laporan lengkap ke file .txt? (y/n): ").lower()

    if simpan == 'y':
        nama_file_txt = f"Laporan_{id_s}.txt"

        with open(nama_file_txt, "w") as f:
            # Menulis konten yang sama persis dengan yang tampil di layar
            f.write("="*60 + "\n")
            f.write(f"LAPORAN ANALISIS SEKUENS LENGKAP: {id_s}\n")
            f.write("="*60 + "\n\n")

            f.write(f"DNA Asli           : {str(my_dna)}\n")
            f.write(f"Reverse Complement : {str(my_dna.reverse_complement())}\n")
            f.write(f"RNA (Transkripsi)  : {str(my_rna)}\n")
            f.write(f"Protein Lengkap    : {str(protein_lengkap)}\n")
            f.write(f"Protein Terpanjang (ORF): {str(protein_orf)}\n\n")

            f.write(f"Persentase GC      : {gc_fraction(my_dna)*100:.2f}%\n")
            f.write(f"Berat Protein      : {molecular_weight(protein_orf, 'protein'):.2f} Da\n")
            f.write(f"Titik Isolistrik   : {pa.isoelectric_point():.2f} (pH)\n\n")

            f.write(f"Lokasi Start (ATG) : {starts}\n")
            f.write(f"Lokasi Stop        : {stops}\n\n")

            f.write("Komposisi Asam Amino (Jumlah):\n")
            for simbol, jumlah in pa.count_amino_acids().items():
                if jumlah > 0:
                    detail = AA_DATABASE.get(simbol, {"nama": "Unknown", "sifat": "Unknown"})
                    f.write(f"  - {detail['nama']} ({simbol}) [{detail['sifat']}]: {int(jumlah)} unit\n")

            f.write("\n" + "="*60 + "\n")
            f.write("Laporan ini dihasilkan secara otomatis oleh Biotech Pipeline.\n")

        print(f"✅ Berhasil! Laporan lengkap telah disimpan sebagai: {nama_file_txt}")
        print("Cek folder di sebelah kiri Google Colab untuk mengunduhnya.")

    if input("\nSimpan hasil ke file FASTA? (y/n): ").lower() == 'y':
        # Simpan FASTA Protein
        SeqIO.write(SeqRecord(protein_orf, id=id_s, description="ORF_Protein"), f"{id_s}.fasta", "fasta")
        print(f"✅ File {id_s}_laporan.csv dan {id_s}.fasta berhasil disimpan!")

# Jalankan Program
if __name__ == "__main__":
    jalankan_pipeline_lengkap()

=== AUTOMATED PIPELINE ===

[1] TIPE ORGANISME
1. Standard (Eukariota/Umum)
11. Bakteri, Archaea, & Plastida
Pilih nomor tabel (1/11): 1

[2] METODE INPUT
1. Manual | 2. File FASTA | 3. NCBI
Pilih metode: 3
Masukkan Accession ID: PI453901

Ingin cek mutasi terhadap referensi? (y/n): n

LAPORAN ANALISIS SEKUENS: PI453901.1
DNA Asli          : ATATTAGGTTTATACCTTCCCAGGTAACAAACCAACCAACTTTCGATCTCTTGTAGATCT...
Reverse Complement: TTTTTTTTTTTTTTTTTTTTTTTTTTTTTGTCATTCTCCTAAGAAGCTATTAAAATCACA...
RNA (Transkripsi) : AUAUUAGGUUUAUACCUUCCCAGGUAACAAACCAACCAACUUUCGAUCUCUUGUAGAUCU...
Protein Lengkap   : ILGLYLPR*QTNQLSISCRSVL*TNFKICVAVTRLHA*CTHAV*LITNYCR*QDTSNSSI...
Protein Terpanjang (ORF): MESLVPGFNEKTHVQLSLPVLQVRDVLVRGFGDSVEEVLSEARQHLKDGTCGLVEVEKGV...

Persentase GC     : 3795.11%
Berat Protein     : 489983.45 Da
Titik Isolistrik  : 6.04 (pH)

Lokasi Start (ATG): [106, 265, 407, 467, 488, 506, 512, 517, 593, 725, 784, 818, 845, 900, 965, 971, 1010, 1064, 1071, 1145, 1153, 1193, 1197, 1207, 1225, 1